# SLP Project — End-to-End Pipeline Test
Runs the full pipeline on a single video: MediaPipe lip extraction → AV-HuBERT teacher inference → Student distillation training step.

**Runtime:** GPU (A100 recommended). Go to Runtime → Change runtime type → GPU.

## 1. Setup & Dependencies

In [ ]:
# Clone av-hubert and install fairseq
!git clone https://github.com/facebookresearch/av_hubert.git
%cd av_hubert
!git submodule init && git submodule update
!pip install -r requirements.txt
%cd fairseq
!pip install --editable ./
%cd ../..

In [ ]:
# Install remaining dependencies
# Colab is Python 3.12: mediapipe 0.10.5 has no wheel, opencv 4.8 is too old
!pip install "numpy>=1.24,<1.27" mediapipe opencv-python torchvision

In [ ]:
# Fix omegaconf/fairseq compatibility — pin version that still exports 'II'
!pip install "omegaconf>=2.0,<2.4" "hydra-core>=1.0,<1.4"

In [ ]:
# Download pretrained AV-HuBERT checkpoint (Large, LRS3 + VoxCeleb2)
!wget -q -O checkpoint.pt https://dl.fbaipublicfiles.com/avhubert/model/lrs3_vox/avsr/large_noise_pt_noise_ft_433h.pt
!ls -lh checkpoint.pt

In [ ]:
# Download sample video from Oxford-BBC LRW
!wget -q -O AFTERNOON.mp4 https://www.robots.ox.ac.uk/~vgg/data/lip_reading/data/AFTERNOON.mp4
!ls -lh AFTERNOON.mp4

In [ ]:
# Download mediapipe face landmarker model
import urllib.request
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
    "face_landmarker.task"
)
print("face_landmarker.task downloaded")

import sys
import dataclasses
import copy
from pathlib import Path

# Monkeypatch: fairseq uses mutable dataclass defaults, which Python 3.12
# rejects. This relaxes the check by auto-wrapping them in default_factory.
_original_get_field = dataclasses._get_field

def _relaxed_get_field(cls, a_name, a_type, default_kw_only):
    try:
        return _original_get_field(cls, a_name, a_type, default_kw_only)
    except ValueError as e:
        if "mutable default" not in str(e):
            raise
        default = getattr(cls, a_name)
        setattr(cls, a_name, dataclasses.field(default_factory=lambda d=default: copy.deepcopy(d)))
        return _original_get_field(cls, a_name, a_type, default_kw_only)

dataclasses._get_field = _relaxed_get_field

FAIRSEQ_PATH = Path("av_hubert/fairseq")
sys.path.insert(0, str(FAIRSEQ_PATH))

AV_HUBERT_PATH = Path("av_hubert/avhubert")
sys.path.insert(0, str(AV_HUBERT_PATH))

import os
import math
import logging
import warnings
import tempfile

import numpy as np
import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

import fairseq  # type: ignore
from fairseq import tasks, utils
from fairseq.dataclass.configs import GenerationConfig
from omegaconf import OmegaConf

import hubert_pretraining, hubert, hubert_asr  # type: ignore

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("hubert_pretraining").setLevel(logging.WARNING)
logging.getLogger("hubert").setLevel(logging.WARNING)
logging.getLogger("hubert_dataset").setLevel(logging.WARNING)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import sys
from pathlib import Path

FAIRSEQ_PATH = Path("av_hubert/fairseq")
sys.path.insert(0, str(FAIRSEQ_PATH))

AV_HUBERT_PATH = Path("av_hubert/avhubert")
sys.path.insert(0, str(AV_HUBERT_PATH))

import os
import math
import logging
import warnings
import tempfile

import numpy as np
import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

import fairseq  # type: ignore
from fairseq import tasks, utils
from fairseq.dataclass.configs import GenerationConfig
from omegaconf import OmegaConf

import hubert_pretraining, hubert, hubert_asr  # type: ignore

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("hubert_pretraining").setLevel(logging.WARNING)
logging.getLogger("hubert").setLevel(logging.WARNING)
logging.getLogger("hubert_dataset").setLevel(logging.WARNING)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. MediaPipe Lip Extraction (mp.py)

In [ ]:
def extract_lip_crop(frame, detector, target_size=(96, 96)):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    result = detector.detect(mp_image)
    if not result.face_landmarks:
        return None
    landmarks = result.face_landmarks[0]
    h, w = frame.shape[:2]
    LIP_INDICES = [
        61, 146, 91, 181, 84, 17, 314, 405, 321, 375,
        291, 308, 324, 318, 402, 317, 14, 87, 178, 88,
        95, 78, 191, 80, 81, 82, 13, 312, 311, 310, 415
    ]
    lip_points = []
    for idx in LIP_INDICES:
        lm = landmarks[idx]
        lip_points.append((int(lm.x * w), int(lm.y * h)))
    lip_points = np.array(lip_points)
    x_min, y_min = lip_points.min(axis=0)
    x_max, y_max = lip_points.max(axis=0)
    pad_x = int((x_max - x_min) * 0.3)
    pad_y = int((y_max - y_min) * 0.3)
    x_min = max(0, x_min - pad_x)
    y_min = max(0, y_min - pad_y)
    x_max = min(w, x_max + pad_x)
    y_max = min(h, y_max + pad_y)
    crop = frame[y_min:y_max, x_min:x_max]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, target_size)
    return resized


def process_video(video_path):
    options = vision.FaceLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path="face_landmarker.task"),
        num_faces=1,
        min_face_detection_confidence=0.5,
    )
    detector = vision.FaceLandmarker.create_from_options(options)
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"ERROR: Could not open {video_path}.")
        return np.array([]), 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps == 0 or frame_count == 0:
        print(f"ERROR: Video has 0 fps or 0 frames.")
        cap.release()
        return np.array([]), 0

    print(f"Video: {fps:.1f} fps, {frame_count} frames")
    crops = []
    failed = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        crop = extract_lip_crop(rgb, detector)
        if crop is not None:
            crops.append(crop)
        else:
            failed += 1
    cap.release()
    detector.close()
    total = len(crops) + failed
    if total > 0:
        print(f"Extracted {len(crops)}/{total} frames ({failed} failed, {failed/total*100:.1f}% drop rate)")
        if failed / total > 0.20:
            print("WARNING: >20% drop rate")
        if abs(fps - 25) > 1:
            print(f"WARNING: {fps:.0f}fps recorded, model expects 25fps")
    return np.array(crops) if crops else np.array([]), fps


def save_crops_as_video(crops, output_path, fps=25):
    h, w = crops[0].shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h), isColor=False)
    for frame in crops:
        out.write(frame)
    out.release()
    print(f"Saved {len(crops)} frames to {output_path}")


def plot_crops(crops):
    if len(crops) > 0:
        num_show = min(8, len(crops))
        fig, axes = plt.subplots(1, num_show, figsize=(16, 3))
        for i, ax in enumerate(axes):
            idx = i * (len(crops) // num_show)
            ax.imshow(crops[idx], cmap='gray')
            ax.axis('off')
            ax.set_title(f"F{idx}")
        plt.suptitle(f"Lip crops ({len(crops)} frames)")
        plt.show()

## 4. Teacher Utilities (model_utils.py)

In [ ]:
def prep_inference(video_path, model, cfg, task):
    num_frames = int(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_COUNT))
    data_dir = tempfile.mkdtemp()
    tsv_cont = ["/\n", f"test-0\t{video_path}\t{None}\t{num_frames}\t{int(16_000*num_frames/25)}\n"]
    label_cont = ["DUMMY\n"]
    with open(f"{data_dir}/test.tsv", "w") as fo:
        fo.write("".join(tsv_cont))
    with open(f"{data_dir}/test.wrd", "w") as fo:
        fo.write("".join(label_cont))

    gen_subset = "test"
    gen_cfg = GenerationConfig(beam=20)
    cfg_dict = OmegaConf.to_container(cfg, resolve=True)
    cfg_dict["task"]["modalities"] = ["video"]
    cfg_dict["task"]["data"] = data_dir
    cfg_dict["task"]["label_dir"] = data_dir
    cfg_dict["task"]["noise_prob"] = 0.0
    cfg_dict["task"]["noise_wav"] = None
    cfg = OmegaConf.create(cfg_dict)
    task = tasks.setup_task(cfg.task)
    task.load_dataset(gen_subset, task_cfg=cfg.task)
    generator = task.build_generator([model], gen_cfg)

    def hypo_token_decoder(x):
        dictionary = task.target_dictionary
        symbols_ignore = generator.symbols_to_strip_from_output
        symbols_ignore.add(dictionary.pad())
        return task.datasets[gen_subset].label_processors[0].decode(x, symbols_ignore)

    itr = task.get_batch_iterator(dataset=task.dataset(gen_subset)).next_epoch_itr(shuffle=False)
    return itr, generator, hypo_token_decoder


def run_inference_and_extract_soft_targets(model, itr, temperature=1.0):
    model.num_updates = 999999
    with torch.no_grad():
        sample = next(itr)
        net_output = model(**sample['net_input'])
        logits = net_output[0]
        return torch.softmax(logits / temperature, dim=-1)


def crops_to_tensor(crops):
    frames = crops.astype(np.float32) / 255.0
    frames = (frames - 0.421) / 0.165
    tensor = torch.FloatTensor(frames).unsqueeze(0).unsqueeze(2)
    return tensor

## 5. Student Model (student_model.py)

In [ ]:
class ResNet18Frontend(nn.Module):
    """
    ResNet-18 pretrained on ImageNet, modified for grayscale mouth ROI frames.
    Input: (batch, num_frames, 1, 88, 88)
    Output: (batch, num_frames, embed_dim)
    """
    def __init__(self, embed_dim=256, freeze_early=True):
        super().__init__()
        resnet = models.resnet18(pretrained=True)

        pretrained_weight = resnet.conv1.weight.data
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        resnet.conv1.weight.data = pretrained_weight.mean(dim=1, keepdim=True)

        self.features = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
            resnet.avgpool,
        )

        if freeze_early:
            for name, param in self.features.named_parameters():
                if not name.startswith('6.') and not name.startswith('7.'):
                    param.requires_grad = False

        self.proj = nn.Linear(512, embed_dim)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        x = self.features(x)
        x = x.view(B * T, -1)
        x = self.proj(x)
        x = x.view(B, T, -1)
        return x


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class StudentLipReader(nn.Module):
    def __init__(
        self,
        vocab_size=1000,
        embed_dim=256,
        encoder_layers=4,
        decoder_layers=4,
        n_heads=4,
        ff_dim=512,
        dropout=0.1,
        pad_idx=1,
        freeze_early_resnet=True,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.pad_idx = pad_idx

        self.visual_frontend = ResNet18Frontend(
            embed_dim=embed_dim,
            freeze_early=freeze_early_resnet,
        )

        self.pos_encoder = PositionalEncoding(embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=encoder_layers)

        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_decoder = PositionalEncoding(embed_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=decoder_layers)

        self.output_proj = nn.Linear(embed_dim, vocab_size)

    def _make_causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
        return mask

    def encode(self, video_frames):
        x = self.visual_frontend(video_frames)
        x = self.pos_encoder(x)
        x = self.encoder(x)
        return x

    def decode(self, encoder_out, prev_tokens):
        x = self.token_embedding(prev_tokens)
        x = self.pos_decoder(x)
        causal_mask = self._make_causal_mask(prev_tokens.size(1), prev_tokens.device)
        x = self.decoder(x, encoder_out, tgt_mask=causal_mask)
        logits = self.output_proj(x)
        return logits

    def forward(self, video_frames, prev_tokens):
        encoder_out = self.encode(video_frames)
        logits = self.decode(encoder_out, prev_tokens)
        return logits

    def count_parameters(self, only_trainable=False):
        if only_trainable:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

    def print_parameter_breakdown(self):
        components = {
            'ResNet-18 frontend': self.visual_frontend,
            '  resnet features': self.visual_frontend.features,
            '  projection (512->embed)': self.visual_frontend.proj,
            'Transformer encoder (4 layers)': self.encoder,
            'Token embedding': self.token_embedding,
            'Transformer decoder (4 layers)': self.decoder,
            'Output projection': self.output_proj,
        }
        print(f"{'Component':<35} {'Total':>12} {'Trainable':>12}")
        print("-" * 61)
        for name, module in components.items():
            total = sum(p.numel() for p in module.parameters())
            trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
            print(f"{name:<35} {total:>12,} {trainable:>12,}")
        print("-" * 61)
        total = self.count_parameters(only_trainable=False)
        trainable = self.count_parameters(only_trainable=True)
        frozen = total - trainable
        print(f"{'TOTAL':<35} {total:>12,} {trainable:>12,}")
        print(f"{'FROZEN':<35} {'':>12} {frozen:>12,}")


class DistillationTrainer:
    def __init__(self, student, temperature=2.0, alpha=0.7, lr=1e-3):
        self.student = student
        self.temperature = temperature
        self.alpha = alpha
        self.optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, student.parameters()),
            lr=lr,
        )

    def compute_loss(self, student_logits, teacher_soft_targets, hard_targets):
        T = self.temperature

        student_log_probs = F.log_softmax(student_logits / T, dim=-1)
        soft_loss = F.kl_div(
            student_log_probs,
            teacher_soft_targets,
            reduction='batchmean',
        ) * (T ** 2)

        hard_loss = F.cross_entropy(
            student_logits.view(-1, student_logits.size(-1)),
            hard_targets.view(-1),
            ignore_index=self.student.pad_idx,
        )

        combined = self.alpha * soft_loss + (1 - self.alpha) * hard_loss
        return combined, soft_loss, hard_loss

    def train_step(self, video_frames, prev_tokens, teacher_soft_targets, hard_targets):
        self.student.train()
        self.optimizer.zero_grad()

        student_logits = self.student(video_frames, prev_tokens)
        loss, soft_loss, hard_loss = self.compute_loss(
            student_logits, teacher_soft_targets, hard_targets
        )

        loss.backward()
        self.optimizer.step()

        return {
            'loss': loss.item(),
            'soft_loss': soft_loss.item(),
            'hard_loss': hard_loss.item(),
        }

## 6. Load Teacher Model

In [ ]:
ckpt_path = "checkpoint.pt"
models_list, cfg, task = fairseq.checkpoint_utils.load_model_ensemble_and_task([ckpt_path])
teacher_model = models_list[0].eval()
print(f"Teacher model loaded: {sum(p.numel() for p in teacher_model.parameters()):,} parameters")

## 7. Extract Lip Crops

In [ ]:
video_path = "AFTERNOON.mp4"
crops, fps = process_video(video_path)
plot_crops(crops)
save_crops_as_video(crops, "AFTERNOON-roi.mp4")

## 8. Teacher Inference → Soft Targets

In [ ]:
itr, generator, hypo_token_decoder = prep_inference(
    os.path.abspath("AFTERNOON-roi.mp4"), teacher_model, cfg, task
)

soft_targets = run_inference_and_extract_soft_targets(teacher_model, itr)
print(f"Soft targets shape: {soft_targets.shape}")
print(f"  → batch={soft_targets.size(0)}, seq_len={soft_targets.size(1)}, vocab={soft_targets.size(-1)}")

## 9. Wire Up Real Training Data

In [ ]:
# Derive dimensions from real teacher output
VOCAB_SIZE = soft_targets.size(-1)
SEQ_LEN = soft_targets.size(1)
BATCH_SIZE = 1

# Get special token indices from teacher's dictionary
target_dict = task.target_dictionary
PAD_IDX = target_dict.pad()
BOS_IDX = target_dict.eos()  # fairseq uses EOS as BOS for decoder input

print(f"Vocab size: {VOCAB_SIZE}")
print(f"Seq len:    {SEQ_LEN}")
print(f"PAD index:  {PAD_IDX}")
print(f"BOS index:  {BOS_IDX}")

# Derive hard targets from teacher's best predictions
hard_targets = soft_targets.argmax(dim=-1)
print(f"\nHard targets shape: {hard_targets.shape}")
print(f"Hard targets:       {hard_targets[0].tolist()}")

# Construct prev_tokens: [BOS, tok_0, tok_1, ...] (shifted right for teacher forcing)
prev_tokens = torch.cat([
    torch.full((BATCH_SIZE, 1), BOS_IDX, dtype=torch.long),
    hard_targets[:, :-1],
], dim=1)
print(f"Prev tokens shape:  {prev_tokens.shape}")
print(f"Prev tokens:        {prev_tokens[0].tolist()}")

# Prepare video frames
video_frames = crops_to_tensor(crops)
print(f"\nVideo frames shape: {video_frames.shape}")
print(f"  → batch={video_frames.size(0)}, frames={video_frames.size(1)}, C={video_frames.size(2)}, H={video_frames.size(3)}, W={video_frames.size(4)}")

## 10. Initialize Student & Run Training Step

In [ ]:
student = StudentLipReader(
    vocab_size=VOCAB_SIZE,
    embed_dim=256,
    encoder_layers=4,
    decoder_layers=4,
    n_heads=4,
    ff_dim=512,
    pad_idx=PAD_IDX,
    freeze_early_resnet=True,
)
student.print_parameter_breakdown()

In [ ]:
trainer = DistillationTrainer(student, temperature=2.0, alpha=0.7)
losses = trainer.train_step(video_frames, prev_tokens, soft_targets, hard_targets)

print(f"Combined loss: {losses['loss']:.4f}")
print(f"Soft loss:     {losses['soft_loss']:.4f}")
print(f"Hard loss:     {losses['hard_loss']:.4f}")
print("\n✅ End-to-end pipeline works. Real teacher soft targets flowing into student.")